# Week 4 — Wednesday: Joining Tables — Keys and Relationships

**DATA 202 · Calvin University**

Monday's question was *"what is one row about?"* Today we ask it of **two tables at once** — and then ask the harder one:

> 🧭 **Do these two rows describe the *same thing*?**

A join is a **claim** that a row in one table and a row in another are about the same thing. `pd.merge()` can't check that claim — it only checks whether two key values are *equal*. Checking the claim is our job.

**Setup:** a biomaterials lab. Five tables, from different tests on (mostly) the same materials — some joins between them are great, some run without error and mean nothing.

**Today's plan (50 min):**

| Time | Part |
|---|---|
| ~4 min | Load the five tables and map them |
| ~8 min | **Part 1 — Keys and Relational Structure** (SLO 04A) |
| ~13 min | **Part 2 — The Four Join Types** (SLO 04B) |
| ~20 min | **Part 3 — Does This Join Make Sense?** (SLO 04B) |
| ~5 min | Careful with joining + what's next |

**Cues:** 🧭 Join check · 💬 Ask the class · 🔨 Task

---
## Loading the Tables · ~4 min

In [ ]:
import pandas as pd

BASE = "https://cs.calvin.edu/courses/data/202/fsantos/biomaterial/"

properties             = pd.read_csv(BASE + "biomaterial_properties.csv")
biocompatibility       = pd.read_csv(BASE + "biomaterial_biocompatibility.csv")
degradation            = pd.read_csv(BASE + "biomaterial_degradation.csv")
environment_conditions = pd.read_csv(BASE + "environment_conditions.csv")
biocompatibility_time  = pd.read_csv(BASE + "biocompatibility_time.csv")

tables = {
    "properties": properties,
    "biocompatibility": biocompatibility,
    "degradation": degradation,
    "environment_conditions": environment_conditions,
    "biocompatibility_time": biocompatibility_time,
}
for name, t in tables.items():
    print(f"--- {name}  {t.shape}")
    display(t.head(3))

Monday's question, asked of each table:

| Table | One row is about… | Identifying column(s) |
|---|---|---|
| `properties` | one **material** and its mechanical properties | `Material ID` |
| `biocompatibility` | one **cell test** of a material (one condition, one duration in *days*) | `Material ID` |
| `biocompatibility_time` | one material's cell viability at day 7, 14 and 28 *(a wide table!)* | `Material ID` + `Condition` |
| `degradation` | one **degradation test**: a material sitting in an environment for some *weeks* | `Material ID` + `Environment` |
| `environment_conditions` | one **environment** (its pH, temperature, composition) | `Environment` |

Sizes: 8, 8, 8, 13 and 6 rows. Five tables, **different tests, different units, different durations**.

---
## Part 1 — Keys and Relational Structure (SLO 04A) · ~8 min

To connect a row in one table to a row in another we need a column that names **which** thing, identically, in both. That column is a **key**.

* **Primary key** — uniquely identifies each row in *its own* table. `Material ID` in `properties`; `Environment` in `environment_conditions`.
* **Foreign key** — a column that *points to* a primary key in another table: "this row is about that thing, over there."

```
properties ──── Material ID ────┬──── biocompatibility
 (8 materials)                  ├──── biocompatibility_time
                                └──── degradation ──── Environment ──── environment_conditions
```

A key only works if the values match **exactly** — `pd.merge()` compares text, not meaning.

> 💬 **Ask the class:** In `degradation`, which columns are foreign keys — and what do they point to?

<details><summary>Answer</summary>

`Material ID` → `properties`, and `Environment` → `environment_conditions`. `degradation` is the table in the middle: it connects materials to environments.
</details>

> 💬 **Ask the class:** Is `Environment` unique in `environment_conditions`? Is it unique in `degradation`?

<details><summary>Answer</summary>

Unique in `environment_conditions` (one row per environment — its primary key). **Not** unique in `degradation` (several materials are tested in Aqueous) — so environments *repeat* there. Run the cell to see.
</details>

In [ ]:
print("Material ID unique in properties?             ", properties["Material ID"].is_unique)
print("Environment unique in environment_conditions? ", environment_conditions["Environment"].is_unique)
print("Environment unique in degradation?            ", degradation["Environment"].is_unique)
degradation["Environment"].value_counts()

`properties` is the master list of materials. So every other table's `Material ID` should be found there.


> 💬 **Ask the class:** Does every `Material ID` in the other three tables appear in `properties`? Does every material in `properties` appear in every other table? Guess, then run.

<details><summary>Answer</summary>

No, on both counts — see the output and the note below.
</details>

In [ ]:
known = set(properties["Material ID"])

for name in ["biocompatibility", "degradation", "biocompatibility_time"]:
    ids = set(tables[name]["Material ID"])
    print(f"{name:22s} IDs missing from properties: {sorted(ids - known)}")
    print(f"{'':22s} materials never tested here: {sorted(known - ids)}")

* `degradation` has six materials (`B009`–`B014`) and `biocompatibility` has two (`B009`, `B010`) with **no row in `properties`** — tests on materials we have no properties for.
* `B004` (Collagen) was never degradation-tested; `B005` and `B008` (the two metal alloys) were never cell-tested.

A missing partner row means **"no record"** — not "the material doesn't exist" and not "the test failed."

---
### 🔨 Mini-Task A — Find the Orphans With `.isin()` (~3 min)

1. Filter `degradation` to rows whose `Material ID` is **not** in `properties["Material ID"]` → `deg_orphans`
2. Filter `properties` to rows whose `Material ID` is **not** in `degradation["Material ID"]` → `props_untested`

<details><summary>Hint</summary>

`df[~df["col"].isin(other_df["col"])]` — the `~` flips "is in" to "is not in".
</details>

<details><summary>Check yourself</summary>

`deg_orphans` has 6 rows (`B009`–`B014`); `props_untested` has 1 row (`B004`, Collagen) — same as the set check above.
</details>

In [ ]:
# Your code here

---
## Part 2 — The Four Join Types (SLO 04B) · ~13 min

`pd.merge()` combines two tables on a shared key. `how=` decides what happens to a row that *doesn't* find a match:

| `how=` | Keeps | Unmatched rows get... |
|:---|:---|:---|
| `"inner"` | only rows matched in **both** tables | dropped completely, from both sides |
| `"left"` | every row from the **left** table | `NaN` filled in for the right table's columns |
| `"right"` | every row from the **right** table | `NaN` filled in for the left table's columns |
| `"outer"` | every row from **either** table | `NaN` filled in on whichever side is missing |

We start with the most natural pair: what a material *is* (`properties`) and how cells *react* to it (`biocompatibility`).

> 🧭 **Join check**
> - *What is one row about, in each table?* → `properties`: one material · `biocompatibility`: one cell test of a material
> - *Do the two rows describe the same thing — same material, same experiment, same conditions?* → yes — the same material (`Material ID`), and a material's mechanical properties don't depend on the test
> - *What will one row of the result be about?* → one material with one of its cell tests

> 💬 **Ask the class:** From Part 1: **6** materials are in both tables, **2** (`B005`, `B008`) only in `properties`, **2** (`B009`, `B010`) only in `biocompatibility`. Predict the row count for `inner`, `left`, `right` and `outer`.

<details><summary>Answer</summary>

inner **6** · left **6 + 2 = 8** · right **6 + 2 = 8** · outer **6 + 2 + 2 = 10**.
</details>

In [ ]:
for how in ["inner", "left", "right", "outer"]:
    joined = pd.merge(properties, biocompatibility, on="Material ID", how=how)
    print(f"{how:6s}", joined.shape)

Same two tables, four different answers. `indicator=True` adds a `_merge` column that says where each row came from — the quickest way to see who found a partner:

In [ ]:
outer = pd.merge(properties, biocompatibility, on="Material ID", how="outer", indicator=True)
outer[["Material ID", "Material Name", "Cell Viability (%)", "_merge"]]

> 💬 **Ask the class:** Which materials are `left_only`? `right_only`? What does the `NaN` in `Cell Viability (%)` for a `left_only` row mean — zero viability, or something else?

<details><summary>Answer</summary>

`left_only`: `B005` Titanium Alloy and `B008` Magnesium Alloy — never cell-tested. `right_only`: `B009`, `B010` — tests on materials with no properties on record. The `NaN` means **unknown / no record** — *not* zero, and *not* "failed."
</details>

---
### 🔨 Mini-Task B — Pick the Right Join (~3 min)

The materials team wants **every material we have mechanical data for, with cell-test results filled in where they exist**. Which `how=` gives that in one call? Assign to `props_with_bio` and check the shape.

<details><summary>Hint</summary>

"Every material we have mechanical data for" — which table has to keep *all* of its rows? Put it on the left.
</details>

<details><summary>Check yourself</summary>

`how="left"`, shape `(8, 9)`: all 8 materials in `properties`, 4 columns of `biocompatibility` added, `NaN` for `B005` and `B008`.
</details>

In [ ]:
# Your code here
props_with_bio = None

---
### 🔨 Task — Strong *and* Biocompatible? (~6 min)

**Real question:** which materials are both mechanically strong (`Tensile Strength (MPa)` **> 40**) and biocompatible (`Cell Viability (%)` **> 80**)?

1. **Join** `properties` and `biocompatibility` with the `how=` that keeps every material → `combined` (same choice as Mini-Task B)
2. **Filter** `combined` on *both* conditions → `strong_safe`. Which materials pass?
3. **Who never got judged?** Filter `combined` for rows with tensile strength > 40 **and a missing** `Cell Viability (%)` → `strong_untested`. (Hint: `.isna()`)

<details><summary>Hint</summary>

Combine conditions with `&` and wrap each in parentheses: `df[(df["a"] > 1) & (df["b"] > 2)]`. For missing values: `df["b"].isna()`.
</details>

<details><summary>Check yourself</summary>

`strong_safe`: **PLA** (50 MPa, 90 %) and **HA** (120 MPa, 95 %). `strong_untested`: **Titanium Alloy** (950 MPa!) and **Magnesium Alloy** (220 MPa) — strong, but never cell-tested. Step 2 silently dropped them (`NaN > 80` is `False`) — *not because they failed, but because nobody tested them.*
</details>

In [ ]:
# Your code here

---
## Part 3 — Does This Join Make Sense? (SLO 04B) · ~20 min

Every join so far *ran*. That's not the same as *making sense*. Before any `pd.merge()`, ask the three questions:

1. **Is the key really the same thing in both tables** — and unique where I think it is?
2. **Do the rows describe the same experiment** — same conditions, same units, same duration?
3. **What will one row of the result be about?** Does that sentence even make sense?

Three tools to check *after* merging: `indicator=True` (who found a partner?), `validate=` (is the relationship what I think?), and comparing `.shape` with the inputs.

### Case 1 — A join that makes sense

> 🧭 **Join check**
> - *What is one row about, in each table?* → `degradation`: one degradation test · `environment_conditions`: one environment
> - *Do the two rows describe the same thing — same material, same experiment, same conditions?* → yes — `Environment` in `degradation` is a foreign key to the primary key in `environment_conditions`; it's a lookup
> - *What will one row of the result be about?* → one degradation test, now with its environment's pH, temperature and composition

Many tests share one environment ("many-to-one"). `validate="many_to_one"` makes pandas *check* that — it raises an error if `Environment` were repeated in `environment_conditions`.

In [ ]:
deg_env = pd.merge(degradation, environment_conditions, on="Environment",
                   how="left", validate="many_to_one", indicator=True)
print(deg_env.shape)
deg_env["_merge"].value_counts()

13 rows in, 13 rows out, and every row is `both` — a clean lookup. (There is exactly one `NaN` in the result: the `pH Range` for Air, which the CSV literally spells `N/A` and pandas reads as missing. Not a failed join — `_merge` proves it.)

### Case 2 — Same key, different experiment

`biocompatibility` and `biocompatibility_time` both report cell viability for a material. Let's join on `Material ID` alone:

> 🧭 **Join check**
> - *What is one row about, in each table?* → `biocompatibility`: one cell test · `biocompatibility_time`: one material's viability at days 7/14/28
> - *Do the two rows describe the same thing — same material, same experiment, same conditions?* → **maybe** — same material, but is it the same *condition*? Both tables have a `Condition` column…
> - *What will one row of the result be about?* → one material with viability from … which experiment?

> 💬 **Ask the class:** If we merge on `Material ID` alone, should we trust every row? What could go wrong?

<details><summary>Answer</summary>

Only if the two tables tested under the same `Condition`. Let's look at whether they did.
</details>

In [ ]:
same_id = pd.merge(biocompatibility, biocompatibility_time,
                   on="Material ID", suffixes=("_bio", "_time"))
print(same_id.shape)
same_id[same_id["Condition_bio"] != same_id["Condition_time"]][["Material ID", "Condition_bio", "Condition_time"]]

`B006` was tested **in vitro** in one table and **in vivo** in the other; `B007` the opposite. Same key, **different experiment** — merging on `Material ID` alone would pair a petri-dish result with a living-organism result. Adding `Condition` to the key fixes it (and the cost is visible — two materials disappear):

In [ ]:
both = pd.merge(biocompatibility, biocompatibility_time, on=["Material ID", "Condition"])
print(both.shape)
both[["Material ID", "Test Duration (Days)", "Cell Viability (%)",
      "Cell Viability Day 14 (%)", "Cell Viability Day 28 (%)"]]

> 💬 **Ask the class:** `B001` has a viability of 90 % at 14 days in one table and 85 % at day 14 in the other. Same material, same condition, same day — which one is right?

<details><summary>Answer</summary>

The join can't say. Two different test runs gave two numbers; matching keys lined the rows up, they didn't **reconcile** the measurements. Before combining them you'd have to know *why* they differ (replicates? different batches?) — that's a question for the people who ran the tests, not for pandas.
</details>

### Case 3 — Joining on a column that isn't a key

Both tables have a `Condition` column, and pandas will happily join on it:

In [ ]:
by_condition = pd.merge(biocompatibility, biocompatibility_time, on="Condition")
by_condition.shape

Two 8-row tables produced **32** rows. `Condition` isn't an ID — it's a *category* (only two values), so every *in vitro* row gets paired with **every** *in vitro* row on the other side (4 × 4 = 16) and the same for *in vivo*. `B001`'s cell test is now attached to `B005`'s time series: rows that mean nothing.

`validate=` is the seatbelt — ask pandas to check that the relationship is what you think it is:

In [ ]:
try:
    pd.merge(biocompatibility, biocompatibility_time, on="Condition", validate="one_to_one")
except pd.errors.MergeError as e:
    print("MergeError:", str(e).splitlines()[0])

### Case 4 — No shared key at all

Can we attach environment temperatures straight to `properties`?

In [ ]:
try:
    pd.merge(properties, environment_conditions)
except pd.errors.MergeError as e:
    print("MergeError:", e)

No shared column → no direct join. But there **is** a path: `properties` —`Material ID`→ `degradation` —`Environment`→ `environment_conditions`. Follow the foreign keys, one merge at a time:

In [ ]:
chain = (properties
         .merge(degradation, on="Material ID", how="left")
         .merge(environment_conditions, on="Environment", how="left"))
chain.shape

One row = **one material, with its degradation test and that test's environment**. `B004` (Collagen) has `NaN`s all the way across — it was never degradation-tested.

---
### 🔨 Task — Join Court (~6 min)

Three proposed joins. For each: **run it, check the shape, and write a one-sentence verdict** — does it make sense, and what does one row of the result mean? (Ask the three questions from the top of Part 3.)

**Proposal 1.** *"Build a material profile: cell-test results and degradation results for each material."* → `biocompatibility` + `degradation` on `Material ID`, `how="inner"`

**Proposal 2.** *"Find tests that ran for the same amount of time."* → `biocompatibility` (days) + `degradation` (weeks) on test duration. Convert weeks to days first (`× 7`) and merge on that.

**Proposal 3.** *"Attach material names to every degradation test."* → `degradation` + `properties` on `Material ID`, `how="inner"`. Compare the row count to the 13 tests you started with.

<details><summary>Hint</summary>

* Proposal 2: `degradation.assign(**{"Test Duration (Days)": degradation["Test Duration (Weeks)"] * 7})` gives a copy whose duration column has the *same name* as the one in `biocompatibility`.
* Proposal 3: to see who vanished, redo it with `how="left"` (or `"outer"`) and `indicator=True`.
</details>

<details><summary>Check yourself</summary>

1. **(7, 8) — conditional.** Key is valid, but these are two *different experiments* (cells, in vitro/in vivo, days vs. degradation, environment, weeks). Fine for a **material profile** (one row = one material with results from separate tests); *not* fine for combining the measurements as if one experiment (e.g. multiplying viability × degradation rate).
2. **(4, 9) — no.** `B003`/`B004` get paired with `B008`/`B014` only because two tests both happened to last 28 days. Durations aren't keys — pandas returned rows, but there's no relationship in them.
3. **(7, 8) — right key, dangerous `how`.** Six of the 13 degradation tests (`B009`–`B014`) vanish because those materials have no properties on record. Use `how="left"` from `degradation` to keep all 13 and *report* the gap.
</details>

In [ ]:
# Proposal 1


# Proposal 2


# Proposal 3


verdict_1 = ""   # one sentence
verdict_2 = ""
verdict_3 = ""

---
## Careful with Joining · ~5 min

`inner` feels like the "safe" choice — no `NaN`s, everything filled in. But look at what an inner join of `properties` and `biocompatibility` did: it dropped **Titanium Alloy**, the strongest material in the lab (950 MPa). It didn't fail a cell test — it was never *given* one. An `inner`-only report would never even mention it.

Not a pandas bug — a **choice** (`how="inner"`) made once, early, that quietly shapes every number after it. And `NaN` from a join means **"no record"** — not zero, not "bad."

**A join is a claim that two rows describe the same thing.** pandas only checks that two values are equal. Before you merge:

1. Is the key really the same thing in both tables, and unique where I expect?
2. Same experiment — same conditions, units, durations?
3. What will one row of the result be about?

…and after: `indicator=True`, `validate=`, and compare `.shape` to the inputs.

→ this week's reading traces *why* a mismatch like `B006`'s happens, and how a join failure can look identical to genuine absence.

---
## Coming Up

| Topic | What's next |
|---|---|
| This week's reading | Same kind of key mismatch, traced through a *data journey* — who collected it, who cleaned it, who's still missing |
| Practice | Melting, pivoting, joining together, on a new dataset |
| Week 5 | Clustering & Dimensionality Reduction — finding groups the data suggests, instead of ones we choose in advance |